# IA Agente - Tools y Math

## Asistente Matematico con LangChain Tool Calling

### Objetivo
En este laboratorio, construiremos un **agente inteligente** que utiliza **herramientas personalizadas** para resolver problemas matematicos complejos y combinarlos con busquedas en Wikipedia.

#### Conceptos Clave
1. **Tool Calling (@tool)**: Decorador de LangChain para definir herramientas que un LLM puede invocar dinamicamente
2. **ReAct Agent**: Agente que razona sobre que herramienta usar, la invoca, observa el resultado, y decide el siguiente paso
3. **Ollama Local**: LLM ejecutado en local sin dependencias cloud (privacidad + velocidad)
4. **LangGraph**: Framework moderno para orquestar agentes con manejo de estado

#### Herramientas que Construiremos
- **4 Operaciones Matematicas**: sumar, restar, multiplicar, dividir (extraen numeros de texto)
- **Busqueda Wikipedia**: sin API keys, informacion contextual en tiempo real

#### Flujo de Ejecucion
1. Cargar configuracion de `config.ini`
2. Inicializar LLM Ollama local
3. Definir herramientas como decoradores `@tool`
4. Crear agente ReAct con `create_react_agent`
5. Ejecutar consultas de ejemplo en espanol
6. Extender con ejercicio de potencia

---
**Entorno**: Python 3.14 (env-llm-314) | **LLM**: Ollama llama3.1:8b local | **Lenguaje**: 100% Espanol

In [1]:
#%pip install langchain langchain-community langchain-ollama langgraph wikipedia -q

In [2]:
# Cargar configuracion desde config.ini
import configparser
import os
from pathlib import Path

# Leer configuracion
config = configparser.ConfigParser()
config.read('config.ini', encoding='utf-8')

# Crear directorios de logs y outputs si no existen
log_dir = config.get('logging', 'log_dir', fallback='logs')
output_dir = config.get('output', 'output_dir', fallback='outputs')

Path(log_dir).mkdir(exist_ok=True)
Path(output_dir).mkdir(exist_ok=True)

# Imprimir configuracion cargada
print(f"[OK] Configuracion cargada: {config.get('project', 'name')}")
print(f"   Modelo: {config.get('ollama', 'model_name')}")
print(f"   Base URL: {config.get('ollama', 'base_url')}")
print(f"   Lenguaje: {config.get('project', 'language')}")

[OK] Configuracion cargada: IA Agente - Tools y Math
   Modelo: qwen2.5:7b
   Base URL: http://localhost:11434
   Lenguaje: es


In [3]:
# Configurar logging (archivo + consola con rotacion)
import logging
from logging.handlers import RotatingFileHandler
from datetime import datetime

log_level = config.get('logging', 'log_level', fallback='INFO')
max_log_size = config.getint('logging', 'max_log_size', fallback=10) * 1024 * 1024
backup_count = config.getint('logging', 'backup_count', fallback=5)

# Crear logger
logger = logging.getLogger('ai_math_agent')
logger.setLevel(getattr(logging, log_level))

# Handler para archivo (con rotacion)
log_file = os.path.join(log_dir, f"ai_math_{datetime.now().strftime('%Y%m%d')}.log")
file_handler = RotatingFileHandler(log_file, maxBytes=max_log_size, backupCount=backup_count)
file_handler.setFormatter(logging.Formatter(
    '[%(asctime)s] %(name)s - %(levelname)s - %(message)s'
))
logger.addHandler(file_handler)

# Handler para consola
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
logger.addHandler(console_handler)

logger.info("[INICIO] IA Agente - Tools y Math")
print(f"[OK] Logging configurado -> {log_file}")

INFO: [INICIO] IA Agente - Tools y Math


[OK] Logging configurado -> logs\ai_math_20260919.log


In [ ]:
# Inicializar LLM Ollama con parametros desde config.ini
from langchain_ollama import ChatOllama

ollama_base_url = config.get('ollama', 'base_url')
ollama_model = config.get('ollama', 'model_name')
ollama_temperature = config.getfloat('ollama', 'temperature')
ollama_max_tokens = config.getint('ollama', 'max_tokens')

# Inicializar con num_ctx=8192 para evitar truncamiento en tool-calling
# (Leccion aprendida de CrewAI 101 con qwen2.5)
# Agregar keep_alive para evitar recargar modelo entre consultas
# num_predict ahora SI se pasa (antes se leia pero nunca se aplicaba)
llm = ChatOllama(
    model=ollama_model,
    base_url=ollama_base_url,
    temperature=ollama_temperature,
    num_predict=ollama_max_tokens,
    num_ctx=8192,
    keep_alive="30m"
)

logger.info(f"[OK] LLM inicializado: {ollama_model}")
print(f"[OK] LLM Ollama configurado")
print(f"   Modelo: {ollama_model}")
print(f"   Temperature: {ollama_temperature}")
print(f"   Contexto: 8192 tokens")
print(f"   Keep-alive: 30 minutos")

In [ ]:
# Definir 4 herramientas matematicas con @tool
from langchain.tools import tool
import re

@tool
def sumar_numeros(expresion: str) -> float:
    """
    Suma una lista de numeros extraidos de un texto.
    Extrae todos los numeros (enteros y decimales) de la expresion y los suma.
    
    Args:
        expresion: Texto que contiene numeros a sumar (ej: "5 y 3 y 10")
    
    Returns:
        La suma de todos los numeros encontrados
    """
    numeros = re.findall(r'-?\d+\.?\d*', expresion)
    resultado = sum(float(n) for n in numeros)
    logger.info(f"Suma: {expresion} = {resultado}")
    return resultado

@tool
def restar_numeros(expresion: str) -> float:
    """
    Resta una lista de numeros extraidos de un texto (orden secuencial).
    El primer numero es el minuendo; se restan los demas en orden.
    
    Args:
        expresion: Texto con numeros (ej: "10 menos 3 menos 2")
    
    Returns:
        El resultado de las restas secuenciales
    """
    numeros = [float(n) for n in re.findall(r'-?\d+\.?\d*', expresion)]
    if not numeros:
        return 0
    resultado = numeros[0]
    for n in numeros[1:]:
        resultado -= n
    logger.info(f"Resta: {expresion} = {resultado}")
    return resultado

@tool
def multiplicar_numeros(expresion: str) -> float:
    """
    Multiplica una lista de numeros extraidos de un texto.
    
    Args:
        expresion: Texto con numeros (ej: "3 por 4 por 2")
    
    Returns:
        El producto de todos los numeros
    """
    numeros = [float(n) for n in re.findall(r'-?\d+\.?\d*', expresion)]
    resultado = 1
    for n in numeros:
        resultado *= n
    logger.info(f"Multiplicacion: {expresion} = {resultado}")
    return resultado

@tool
def dividir_numeros(expresion: str) -> float:
    """
    Divide numeros extraidos de un texto (orden secuencial).
    El primer numero es el dividendo; se divide por los demas en orden.
    
    Args:
        expresion: Texto con numeros (ej: "100 dividido 5 dividido 2")
    
    Returns:
        El resultado de las divisiones secuenciales
    """
    numeros = [float(n) for n in re.findall(r'-?\d+\.?\d*', expresion)]
    if not numeros or numeros[0] == 0:
        return 0
    resultado = numeros[0]
    for n in numeros[1:]:
        if n != 0:
            resultado /= n
    logger.info(f"Division: {expresion} = {resultado}")
    return resultado

@tool
def calcular_potencia(base_y_exponente: str) -> float:
    """
    Calcula la potencia: extrae base y exponente del texto.
    
    Args:
        base_y_exponente: Texto con numeros (ej: "2 elevado a 3")
    
    Returns:
        El resultado de base^exponente
    """
    numeros = [float(n) for n in re.findall(r'-?\d+\.?\d*', base_y_exponente)]
    if len(numeros) < 2:
        return 0
    base, exponente = numeros[0], numeros[1]
    resultado = base ** exponente
    logger.info(f"Potencia: {base}^{exponente} = {resultado}")
    return resultado

logger.info("[OK] 5 herramientas matematicas definidas (4 operaciones basicas + potencia)")
print("[OK] Herramientas matematicas definidas:")
print("   - sumar_numeros")
print("   - restar_numeros")
print("   - multiplicar_numeros")
print("   - dividir_numeros")
print("   - calcular_potencia")

In [ ]:
# Definir herramienta de busqueda Wikipedia (implementacion directa, sin langchain_community)
import wikipedia

wikipedia.set_lang("es")

@tool
def buscar_wikipedia(query: str) -> str:
    """
    Busca un resumen breve en Wikipedia (idioma espanol).
    Retorna las primeras 3 oraciones del articulo encontrado.
    
    Args:
        query: Termino a buscar en Wikipedia
    
    Returns:
        Resumen breve del articulo (hasta 3 oraciones) o mensaje de error
    """
    try:
        resultado = wikipedia.summary(query, sentences=3)
        logger.info(f"Wikipedia: '{query}' -> encontrado")
        return resultado
    except wikipedia.DisambiguationError as e:
        msg = f"Termino ambiguo '{query}'. Opciones: {', '.join(e.options[:5])}"
        logger.info(f"Wikipedia: '{query}' -> ambiguo")
        return msg
    except wikipedia.PageError:
        msg = f"No se encontro articulo para '{query}' en Wikipedia"
        logger.info(f"Wikipedia: '{query}' -> no encontrado")
        return msg
    except Exception as e:
        msg = f"Error al buscar '{query}' en Wikipedia: {str(e)}"
        logger.error(msg)
        return msg

logger.info("[OK] Herramienta Wikipedia configurada (sin deprecacion)")
print("[OK] Herramienta Wikipedia disponible (busquedas publicas sin API key)")

In [ ]:
# Herramienta de busqueda web via Serper (opcional, requiere API key)
import requests

serper_api_key = config.get('serper', 'api_key', fallback='').strip()
search_tool = None

if serper_api_key:
    @tool
    def buscar_web(query: str) -> str:
        """
        Busca en la web via Serper (Google Search API).
        Retorna los 3 primeros resultados.
        Requiere API key configurada en config.ini [serper] api_key.
        
        Args:
            query: Termino a buscar
        
        Returns:
            Resumen de resultados (titulo + link) o mensaje de error
        """
        try:
            resp = requests.post(
                "https://google.serper.dev/search",
                headers={"X-API-KEY": serper_api_key, "Content-Type": "application/json"},
                json={"q": query},
                timeout=10
            )
            resp.raise_for_status()
            organico = resp.json().get("organic", [])[:3]
            if organico:
                resultado = "\n".join(f"[{r.get('position', i+1)}] {r.get('title', 'Sin titulo')}: {r.get('link', '')}" for i, r in enumerate(organico))
                logger.info(f"Serper: '{query}' -> {len(organico)} resultados")
                return resultado
            else:
                msg = f"Sin resultados en Serper para '{query}'"
                logger.info(msg)
                return msg
        except requests.exceptions.RequestException as e:
            msg = f"Error al buscar en Serper: {str(e)}"
            logger.error(msg)
            return msg
    
    logger.info("[OK] Herramienta Serper configurada (API key presente)")
    print("[OK] Herramienta Serper disponible (busqueda web)")
else:
    logger.info("[INFO] Serper deshabilitada (sin API key en config.ini)")
    print("[INFO] Serper deshabilitada. Para activar, configure [serper] api_key en config.ini")

In [ ]:
# Crear el agente ReAct usando LangChain create_agent (API moderna, reemplaza langgraph.prebuilt.create_react_agent)
from langchain.agents import create_agent

# Agrupar todas las herramientas
tools = [
    sumar_numeros,
    restar_numeros,
    multiplicar_numeros,
    dividir_numeros,
    calcular_potencia,
    buscar_wikipedia
]

# Agregar Serper si esta disponible
if search_tool:
    tools.append(search_tool)

# Crear agente con LangChain (reemplaza create_react_agent deprecado de LangGraph v1.0)
math_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
Eres un asistente matematico experto. Tu tarea es resolver problemas matematicos usando las herramientas disponibles.

Instrucciones:
1. Analiza la consulta del usuario y verifica que sea consistente.
2. Usa las herramientas matematicas (sumar, restar, multiplicar, dividir, calcular potencia) para resolver calculos exactos.
3. Si necesitas informacion contextual (ej. poblacion de un pais, datos complementarios), usa Wikipedia.
4. Si tienes acceso a busqueda web, puedes usarla para informacion adicional.
5. Proporciona el resultado de forma clara y directa.
6. Redondea resultados numericos a 2 decimales cuando sea apropiado.
7. Responde SIEMPRE en espanol.
"""
)

logger.info("[OK] Agente ReAct creado con create_agent (API moderna de LangChain)")
print("[OK] Agente ReAct inicializado con herramientas")

In [ ]:
# Resumen del agente compilado
print(f"[OK] Agente compilado con {len(tools)} herramientas: {[t.name for t in tools]}")

In [ ]:
# Ejecutar consultas de ejemplo en espanol + instrumentacion de benchmarking
import time

print("\n" + "="*60)
print("CONSULTAS DE EJEMPLO - CON TRAZABILIDAD DE TIEMPO/TOKENS")
print("="*60)

# Preparar etiqueta de variante para benchmarking
variant_label = config.get('benchmark', 'variant_label', fallback='').strip()
if not variant_label:
    variant_label = f"{ollama_model}_t{ollama_temperature}_ctx8192"

# Preparar lista de consultas
consultas = [
    "Cuanto es 5 + 3 + 10?",
    "Restame 50 menos 12 menos 8.",
    "Multiplica tres por cuatro por dos.",
    "Cual es la poblacion de Peru? Multiplicala por 0.5"
]

# Ejecutar cada consulta con medicion
resultados = []
metricas = []

for i, consulta in enumerate(consultas, 1):
    print(f"\n[Consulta {i}] {consulta}")
    print("-" * 60)
    
    t0 = time.perf_counter()
    try:
        # Invocar agente
        response = math_agent.invoke({"messages": [("user", consulta)]})
        wall_time = time.perf_counter() - t0
        
        # Extraer respuesta final
        respuesta_final = response["messages"][-1].content
        print(f"Respuesta: {respuesta_final}")
        
        # Extraer metricas de tokens desde los mensajes del LLM
        ai_messages = [m for m in response["messages"] if hasattr(m, "usage_metadata") and m.usage_metadata]
        tokens_entrada = sum(m.usage_metadata.get("input_tokens", 0) for m in ai_messages)
        tokens_salida = sum(m.usage_metadata.get("output_tokens", 0) for m in ai_messages)
        
        # Extraer duracion de inferencia Ollama (en nanosegundos, convertir a segundos)
        eval_duration_ns = 0
        for m in ai_messages:
            if hasattr(m, "response_metadata") and m.response_metadata:
                eval_duration_ns += m.response_metadata.get("eval_duration", 0)
        eval_duration_sec = eval_duration_ns / 1e9
        
        # Guardar metrica de benchmarking
        metrica = {
            "timestamp": datetime.now().isoformat(),
            "variant_label": variant_label,
            "modelo": ollama_model,
            "temperatura": ollama_temperature,
            "consulta": consulta,
            "wall_time_seconds": round(wall_time, 3),
            "llm_inference_seconds": round(eval_duration_sec, 3),
            "tokens_entrada": tokens_entrada,
            "tokens_salida": tokens_salida,
            "tokens_total": tokens_entrada + tokens_salida,
            "num_llm_calls": len(ai_messages),
            "exito": True
        }
        
        # Guardar resultado legible
        resultados.append({
            "numero": i,
            "consulta": consulta,
            "respuesta": respuesta_final
        })
        metricas.append(metrica)
        
        # Log
        logger.info(f"Consulta {i}: {wall_time:.2f}s | Tokens: {tokens_entrada}/{tokens_salida} | {consulta[:50]}...")
    
    except Exception as e:
        wall_time = time.perf_counter() - t0
        print(f"[ERROR] {str(e)}")
        
        # Guardar metrica de error
        metrica = {
            "timestamp": datetime.now().isoformat(),
            "variant_label": variant_label,
            "modelo": ollama_model,
            "temperatura": ollama_temperature,
            "consulta": consulta,
            "wall_time_seconds": round(wall_time, 3),
            "exito": False,
            "error": str(e)
        }
        metricas.append(metrica)
        
        logger.error(f"Error en consulta {i}: {str(e)}")

print("\n" + "="*60)
print(f"[OK] {len(resultados)} consultas ejecutadas exitosamente")
print(f"[OK] Variante: {variant_label}")
print("="*60)

In [ ]:
# Guardar resultados en archivo de salida + persistir metricas JSONL para benchmarking
from datetime import datetime
import json

# Guardar resultados legibles
if config.getboolean('output', 'save_results', fallback=True):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    resultado_file = os.path.join(output_dir, f"resultado_{timestamp}.txt")
    
    with open(resultado_file, 'w', encoding='utf-8') as f:
        f.write(f"===================================================\n")
        f.write(f"RESULTADOS - IA Agente - Tools y Math\n")
        f.write(f"===================================================\n")
        f.write(f"Ejecutado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Modelo: {ollama_model}\n")
        f.write(f"Variante: {variant_label}\n\n")
        
        for resultado in resultados:
            f.write(f"[Consulta {resultado['numero']}]\n")
            f.write(f"Pregunta: {resultado['consulta']}\n")
            f.write(f"Respuesta: {resultado['respuesta']}\n")
            f.write(f"-" * 60 + "\n\n")
    
    print(f"[OK] Resultados guardados en: {resultado_file}")
    logger.info(f"Resultados guardados en: {resultado_file}")

# Guardar metricas de benchmarking (append-only JSONL)
if config.getboolean('benchmark', 'enabled', fallback=True):
    benchmark_file = os.path.join(output_dir, "benchmark.jsonl")
    
    with open(benchmark_file, 'a', encoding='utf-8') as f:
        for metrica in metricas:
            f.write(json.dumps(metrica, ensure_ascii=False) + "\n")
    
    print(f"[OK] Metricas anexadas a: {benchmark_file}")
    print(f"[OK] Consultas trazadas: {len(metricas)}")
    logger.info(f"Metricas anexadas a: {benchmark_file} ({len(metricas)} registros)")
else:
    print("[INFO] Benchmarking deshabilitado en config.ini")

In [ ]:
# Notas finales y proximos pasos
print("\n" + "="*70)
print("[RESUMEN] IA Agente - Tools y Math")
print("="*70)

print("""
CONCEPTOS APRENDIDOS:
[OK] Tool Calling (@tool): Decorador para convertir funciones Python en herramientas del LLM
[OK] ReAct Agent: Razonamiento + Actuacion - el LLM decide que herramienta usar
[OK] LangGraph create_react_agent: API moderna para orquestar agentes (reemplaza initialize_agent)
[OK] Ollama Local: LLM privado, sin API keys, bajo latencia
[OK] Wikipedia Tool: Integracion sin credenciales para busquedas contextuales

PROXIMOS PASOS:
1. Lee MANUAL.md para entender la arquitectura completa
2. Consulta EJEMPLOS.md para mas casos de uso y patrones
3. Extiende con nuevas herramientas (raiz cuadrada, modulo, etc.)
4. Integra agentes multiples (ej. un agente para matematica, otro para busqueda)
5. Aplica reflection: haz que el agente auto-critique sus respuestas

REFERENCIA RAPIDA:
- config.ini: Parametros del LLM, directorios de logs/outputs
- logs/: Historial de ejecuciones (busqueda + depuracion)
- outputs/: Resultados guardados en formato texto
- QUICKSTART.md: Empezar en 3 minutos
- SOLUCION_ERRORES.md: Troubleshooting comun

===================================================================
Laboratorio completado. Listo para experimentar y extender!
===================================================================
""")

logger.info("\n" + "="*70)
logger.info("[OK] IA Agente - Tools y Math - EJECUCION COMPLETADA")
logger.info("="*70)